
# Atelier 2 — Tests statistiques (Excel → Python)

Ce notebook est le **miroir Python** de l’Atelier 2 réalisé sous Excel.

Objectifs :
- Vérifier les résultats obtenus dans Excel
- Découvrir Python **pas à pas**, sans complexité
- Comprendre que Python et Excel racontent la **même histoire statistique**

⚠️ **Règle** : le raisonnement prime sur le code.



## Préparation — Charger les données

1. Depuis Excel, exportez l’onglet **Donnees** en CSV : `donnees.csv`
2. Exportez l’onglet **Avant_Apres_Vibration** en CSV : `avant_apres_vibration.csv`
3. Placez les fichiers dans le même dossier que ce notebook (ou uploadez-les dans Colab)


In [6]:

import pandas as pd
from scipy import stats

# Chargement des données principales
df = pd.read_excel("Atelier_2_tests_statistiques.xlsx")

# Chargement des données avant / après

df_ap = pd.read_excel(
    "Atelier_2_tests_statistiques.xlsx",
    sheet_name="Avant_Apres_Vibration"
)

df.head()

,Periode,Site,Mode_Operatoire,Temperature_C,Pression_bar,Charge_kNm,Vibration_mm_s,Indice_Qualite,Defauts_nb,Arret_min,NonConforme
0,4,Site_A,Mode_1,59.78,3.115,532.5,2.509,87.2,2,22.6,0
1,3,Site_C,Mode_1,61.14,3.022,579.8,2.098,90.0,2,29.9,0
2,3,Site_C,Mode_1,59.45,2.984,480.1,1.706,93.5,0,6.5,0
3,4,Site_A,Mode_3,61.63,3.207,483.4,2.353,85.7,1,52.1,0
4,3,Site_B,Mode_1,63.66,3.201,584.1,2.773,86.0,2,42.4,0



## Mission 1 — Deux groupes indépendants

Question métier :
> Le niveau moyen d’Indice_Qualite est-il différent entre deux groupes indépendants ?

Exemple : **Période 1 vs Période 2**


In [8]:

# Sélection des groupes
# Elle extrait les valeurs de la variable Indice_Qualite pour la période 1 et 2, puis les stocke dans la variable g1, g2.
g1 = df[df['Periode'] == 1]['Indice_Qualite']
g2 = df[df['Periode'] == 2]['Indice_Qualite']

# Test statistique
res = stats.ttest_ind(g1, g2, equal_var=False)
res

TtestResult(statistic=np.float64(-0.9135726706681355), pvalue=np.float64(0.36245421881343026), df=np.float64(145.36341403124382))


## Mission 2 — Avant / Après (apparié)

Question métier :
> Une intervention a-t-elle modifié la vibration ?

On travaille sur **les mêmes unités mesurées deux fois**.


In [9]:

avant = df_ap['Vibration_AVANT']
apres = df_ap['Vibration_APRES']

res = stats.ttest_rel(avant, apres)
res

TtestResult(statistic=np.float64(8.640753026420711), pvalue=np.float64(1.3512071079023325e-10), df=np.int64(39))

In [10]:
df_ap['Vibration_APRES'].head()

0    1.695
1    2.364
2    2.223
3    2.608
4    2.250
Name: Vibration_APRES, dtype: float64


## Mission 3 — Comparer plusieurs groupes (≥ 3)

Question métier :
> La moyenne d’Indice_Qualite dépend-elle de la période (1 à 4) ?


In [11]:
# 1. Récupérer les périodes existantes
periodes = sorted(df['Periode'].unique())

# 2. Créer une liste vide pour stocker les groupes
groups = []

# 3. Extraire l'indice qualité pour chaque période
for p in periodes:
    groupe = df[df['Periode'] == p]['Indice_Qualite']
    groups.append(groupe)

# 4. Test ANOVA (comparaison des moyennes)
res = stats.f_oneway(*groups)

res


F_onewayResult(statistic=np.float64(2.7849741889262085), pvalue=np.float64(0.04094443235745678))

### Test de Tukey pour connaitre la période qui diffère des autres

In [18]:

from statsmodels.stats.multicomp import pairwise_tukeyhsd

# Test de Tukey
tukey = pairwise_tukeyhsd(
    endog=df['Indice_Qualite'],   # variable quantitative
    groups=df['Periode'],         # facteur (périodes/méthodes)
    alpha=0.05
)

# Affichage des résultats
print(tukey)


Multiple Comparison of Means - Tukey HSD, FWER=0.05
group1 group2 meandiff p-adj   lower  upper  reject
---------------------------------------------------
     1      2    0.366 0.7747 -0.6232 1.3552  False
     1      3   0.6703 0.2723 -0.2882 1.6288  False
     1      4   1.0122 0.0299  0.0692 1.9553   True
     2      3   0.3043 0.8477  -0.661 1.2697  False
     2      4   0.6463 0.2963 -0.3038 1.5963  False
     3      4   0.3419 0.7712 -0.5761   1.26  False
---------------------------------------------------



## Mission 4 — Plusieurs facteurs

Question métier :
> Indice_Qualite dépend-il de la période ET du mode opératoire ?

⚠️ En Python débutant, on se limite ici à une **lecture conceptuelle**.
Les calculs détaillés sont faits dans Excel.


In [13]:
import statsmodels.api as sm
from statsmodels.formula.api import ols

# Définition du modèle
modele = ols('Indice_Qualite ~ C(Periode) + C(Mode_Operatoire)', data=df).fit()
#OLS Ordinary Least Square : Méthodes des moindres carrés
# ANOVA sans interaction
anova_sans_interaction = sm.stats.anova_lm(modele)
print("ANOVA sans interaction")
print(anova_sans_interaction)

# Modèle avec interaction
modele_inter = ols(
    'Indice_Qualite ~ C(Periode) + C(Mode_Operatoire) + C(Periode):C(Mode_Operatoire)',
    data=df
).fit()

anova_avec_interaction = sm.stats.anova_lm(modele_inter)
print("\nANOVA avec interaction")
print(anova_avec_interaction)


ANOVA sans interaction
                       df       sum_sq    mean_sq          F    PR(>F)
C(Periode)            3.0    45.336568  15.112189   2.955164  0.032710
C(Mode_Operatoire)    2.0   108.979606  54.489803  10.655393  0.000033
Residual            314.0  1605.740701   5.113824        NaN       NaN

ANOVA avec interaction
                                  df       sum_sq    mean_sq          F  \
C(Periode)                       3.0    45.336568  15.112189   3.010376   
C(Mode_Operatoire)               2.0   108.979606  54.489803  10.854470   
C(Periode):C(Mode_Operatoire)    6.0    59.570380   9.928397   1.977755   
Residual                       308.0  1546.170322   5.020034        NaN   

                                 PR(>F)  
C(Periode)                     0.030434  
C(Mode_Operatoire)             0.000028  
C(Periode):C(Mode_Operatoire)  0.068543  
Residual                            NaN  



## Mission 5 — Dépendance entre catégories

Question métier :
> Le statut NonConforme dépend-il du Site ?


In [14]:
# 1. Créer le tableau de contingence
table = pd.crosstab(df['Site'], df['NonConforme'])

# Afficher le tableau
print("Tableau de contingence :")
print(table)

# 2. Test du Chi-deux
resultat = stats.chi2_contingency(table)

# Afficher le résultat complet
resultat


Tableau de contingence :
NonConforme    0   1
Site                
Site_A       121   1
Site_B        98  13
Site_C        83   4


Chi2ContingencyResult(statistic=np.float64(13.226238866197082), pvalue=np.float64(0.0013426373427379), dof=2, expected_freq=array([[115.1375 ,   6.8625 ],
       [104.75625,   6.24375],
       [ 82.10625,   4.89375]]))


## Mission 6 — Normalité

Question métier :
> Les données sont-elles compatibles avec une loi normale ?


In [15]:
# 1. Test de normalité
resultat = stats.normaltest(df['Indice_Qualite'])

# 2. Affichage du résultat
print("Résultat du test de normalité :")
print(resultat)


Résultat du test de normalité :
NormaltestResult(statistic=np.float64(5.997634308598634), pvalue=np.float64(0.04984599363050895))



## Mission 7 — Adéquation de distribution

Question métier :
> La distribution observée est-elle compatible avec une distribution normale ?

⚠️ Les données doivent être standardisées avant le test.


In [16]:
# 1. Standardisation des données
z = (df['Arret_min'] - df['Arret_min'].mean()) / df['Arret_min'].std()

# 2. Test d’adéquation à une loi normale
resultat = stats.kstest(z, 'norm')

# 3. Affichage du résultat
print("Test d'adéquation à la loi normale :")
print(resultat)


Test d'adéquation à la loi normale :
KstestResult(statistic=np.float64(0.07578899364667635), pvalue=np.float64(0.04803859651418141), statistic_location=np.float64(0.21107597469812597), statistic_sign=np.int8(1))



## Synthèse — Ce qu’il faut retenir

- Python confirme (ou non) ce que vous avez trouvé sous Excel
- Les décisions doivent être **cohérentes** entre outils
- Le plus important n’est pas le code, mais :
  - la question métier
  - le choix du test
  - l’interprétation

✅ **Excel pour comprendre — Python pour automatiser et sécuriser**
